In [1]:
%pip install -U pandas numpy sentence-transformers razdel langchain-text-splitters aiohttp tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
import asyncio
import random
import re
import os
import aiohttp
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from razdel import sentenize

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/lib/python3.10/dist-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
YANDEX_FOLDER_ID = "<FOLDER_ID>"
YANDEX_API_KEY = "<API KEY>"
YANDEX_URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

In [4]:
class SemanticTextChunker:
    def __init__(self, model_name='cointegrated/rubert-tiny2', similarity_threshold=0.5, min_words=200, max_words=350):
        print(f"📥 [Semantic] Загрузка модели эмбеддингов {model_name}...")
        self.encoder = SentenceTransformer(model_name)
        self.similarity_threshold = similarity_threshold
        self.min_words = min_words
        self.max_words = max_words

    def clean_text(self, text):
        text = re.sub(r'\s+', ' ', str(text))
        text = re.sub(r'\[\d+(?:,\s*\d+)*\]', '', text)
        return text.strip()

    def chunk_text(self, text):
        text = self.clean_text(text)
        sentences = [s.text for s in sentenize(text)]
        if len(sentences) <= 1: return [" ".join(sentences)]

        embeddings = self.encoder.encode(sentences, show_progress_bar=False)
        similarities = [cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0] for i in
                        range(len(embeddings) - 1)]

        chunks, current_chunk = [], [sentences[0]]
        current_word_count = len(sentences[0].split())

        for i, sentence in enumerate(sentences[1:]):
            sim = similarities[i]
            word_count = len(sentence.split())

            is_topic_changed = sim < self.similarity_threshold and current_word_count >= self.min_words
            is_too_big = current_word_count + word_count > self.max_words

            if is_topic_changed or is_too_big:
                chunks.append(" ".join(current_chunk))
                current_chunk, current_word_count = [sentence], word_count
            else:
                current_chunk.append(sentence)
                current_word_count += word_count

        if current_chunk: chunks.append(" ".join(current_chunk))
        return chunks

    def process_dataframe(self, df, text_col='text', id_col='article_id'):
        print(f"🧠 [Semantic] Начинаем нарезку {len(df)} статей...")
        chunked_data = []
        for _, row in df.iterrows():
            chunks = self.chunk_text(row[text_col])
            for i, chunk in enumerate(chunks):
                word_count = len(chunk.split())
                if word_count >= 3:
                    chunked_data.append({
                        'doc_id': row[id_col],
                        'chunk_id': f"{row[id_col]}_{i}",
                        'text': chunk,
                        'word_count': word_count,
                        'label': 0,
                        'source_model': 'Human'
                    })
        return pd.DataFrame(chunked_data)

In [11]:
PROMPTS = [
    "Перепиши этот научный текст, сохранив академический стиль, но изменив формулировки:\n{text}",
    "Изложи суть этого фрагмента строгим научным языком, как в диссертации:\n{text}"
]


async def generate_yandex_gpt(session, human_row):
    prompt = random.choice(PROMPTS).format(text=human_row['text'])
    body = {
        "modelUri": f"gpt://{YANDEX_FOLDER_ID}/yandexgpt-lite",
        "completionOptions": {"stream": False, "temperature": 0.6, "maxTokens": 2000},
        "messages": [{"role": "system", "text": "Ты профессиональный научный редактор."},
                     {"role": "user", "text": prompt}]
    }
    headers = {"Authorization": f"Api-Key {YANDEX_API_KEY}", "x-folder-id": YANDEX_FOLDER_ID}

    for attempt in range(3):
        try:
            async with session.post(YANDEX_URL, json=body, headers=headers, ssl=False) as resp:
                if resp.status == 200:
                    result = await resp.json()
                    ai_text = result['result']['alternatives'][0]['message']['text']
                    return {
                        'doc_id': human_row['doc_id'],
                        'chunk_id': human_row['chunk_id'] + "_ai",
                        'text': ai_text,
                        'word_count': len(ai_text.split()),
                        'label': 1,
                        'source_model': 'YandexGPT'
                    }
                await asyncio.sleep(2)
        except Exception:
            await asyncio.sleep(2)
    return None


def save_chunk(chunk_data, filename):
    pd.DataFrame([chunk_data]).to_csv(filename, mode='a', index=False, header=not os.path.exists(filename))


async def run_pipeline(human_chunks_df, filename="final_dataset.csv"):
    # Читаем уже сделанное, если файл есть
    processed_ids = set()
    if os.path.exists(filename):
        processed_ids = set(pd.read_csv(filename)['chunk_id'].unique())

    # Сначала сохраняем человеческие чанки, если их нет
    if not os.path.exists(filename):
        human_chunks_df.to_csv(filename, index=False)
        processed_ids.update(human_chunks_df['chunk_id'].unique())
        print("✅ Человеческие данные записаны.")

    records = human_chunks_df.to_dict('records')
    to_generate = [r for r in records if r['chunk_id'] + "_ai" not in processed_ids]

    print(f"🚀 Нужно сгенерировать: {len(to_generate)} из {len(records)} чанков.")

    async with aiohttp.ClientSession() as session:
        sem = asyncio.Semaphore(5)
        count = 0

        async def worker(row_item):
            nonlocal count
            async with sem:
                ai_chunk = await generate_yandex_gpt(session, row_item)
                if ai_chunk:
                    save_chunk(ai_chunk, filename)
                    count += 1
                    if count % 100 == 0:
                        print(f"\n💾 Сохранено {count} новых ИИ-чанков...")
                return ai_chunk
        
        await asyncio.gather(*(worker(r) for r in to_generate))

In [12]:
async def main():
    # Мок-данные
    file_path = "./cyberleninka_articles_20260307_124155_cleaned.csv"

    try:
        df = pd.read_csv(file_path)

        df = df[['article_id', 'text', 'id', 'title', 'text_clean']]

        df = df.dropna(subset=['text_clean'])

        total_articles = len(df)
        print(f"📄 Исходное количество статей: {total_articles}")

        # Нарезка
        chunker = SemanticTextChunker(min_words=200, max_words=300, similarity_threshold=0.5)
        human_chunks_df = chunker.process_dataframe(df, text_col='text_clean')

        total_human_chunks = len(human_chunks_df)

        print("\n" + "=" * 50)
        print("📊 СТАТИСТИКА НАБОРА ДАННЫХ")
        print(f"Исходных статей: {total_articles}")
        print(f"Человеческих чанков (Label 0): {total_human_chunks}")
        print(f"Среднее кол-во чанков на статью: {total_human_chunks / total_articles:.1f}")
        print("=" * 50 + "\n")

        # Генерация
        await run_pipeline(human_chunks_df)

        # Вывод
        final_df = pd.read_csv("final_dataset.csv")
        ai_count = len(final_df[final_df['label'] == 1])
        human_count = len(final_df[final_df['label'] == 0])

        print("\n" + "=" * 50)
        print("✅ ГЕНЕРАЦИЯ ЗАВЕРШЕНА")
        print(f"Всего строк в датасете: {len(final_df)}")
        print(f"Из них Human (Label 0): {human_count}")
        print(f"Из них AI (Label 1): {ai_count}")
        print(f"Итоговый коэффициент увеличения: {len(final_df) / total_articles:.1f}x")
        print("=" * 50)

    except Exception as e:
        print(f"❌ Ошибка при загрузке файла: {e}")

In [ ]:
await main()

📄 Исходное количество статей: 3112
📥 [Semantic] Загрузка модели эмбеддингов cointegrated/rubert-tiny2...


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 3165.51it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 [Semantic] Начинаем нарезку 3112 статей...

📊 СТАТИСТИКА НАБОРА ДАННЫХ
Исходных статей: 3112
Человеческих чанков (Label 0): 33008
Среднее кол-во чанков на статью: 10.6

🚀 Нужно сгенерировать: 33008 из 33008 чанков.

💾 Сохранено 100 новых ИИ-чанков...

💾 Сохранено 200 новых ИИ-чанков...

💾 Сохранено 300 новых ИИ-чанков...

💾 Сохранено 400 новых ИИ-чанков...

💾 Сохранено 500 новых ИИ-чанков...

💾 Сохранено 600 новых ИИ-чанков...

💾 Сохранено 700 новых ИИ-чанков...

💾 Сохранено 800 новых ИИ-чанков...

💾 Сохранено 900 новых ИИ-чанков...

💾 Сохранено 1000 новых ИИ-чанков...

💾 Сохранено 1100 новых ИИ-чанков...

💾 Сохранено 1200 новых ИИ-чанков...

💾 Сохранено 1300 новых ИИ-чанков...

💾 Сохранено 1400 новых ИИ-чанков...

💾 Сохранено 1500 новых ИИ-чанков...

💾 Сохранено 1600 новых ИИ-чанков...

💾 Сохранено 1700 новых ИИ-чанков...

💾 Сохранено 1800 новых ИИ-чанков...

💾 Сохранено 1900 новых ИИ-чанков...

💾 Сохранено 2000 новых ИИ-чанков...

💾 Сохранено 2100 новых ИИ-чанков...

💾 Сохранено 22